<a href="https://colab.research.google.com/github/thabomosenthal/symmetrical-fortnight/blob/main/Week2_08_embeddings_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2.3: Embeddings Deep Dive

## Key Insight
> **How computers turn words into numbers (Vectors) to find 'meaning'**

## Learning Objectives
By the end of this notebook, you will:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

## Exercises
This notebook contains 5 exercises ranging from easy to advanced.

## Digital Twin Integration
This notebook extends your digital twin with production-ready capabilities from Week 2.

In [1]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Tuple
import re
import json

print('✅ Week 2.3: Embeddings Deep Dive - Ready to start!')
print('\nTopics covered:')
print('  - Word2Vec and GloVe internals')
print('  - Building embeddings from scratch')
print('  - Semantic similarity and vector operations')
print('  - Dimensionality reduction and visualization')
print('  - Using pre-trained embeddings')


✅ Week 2.3: Embeddings Deep Dive - Ready to start!

Topics covered:
  - Word2Vec and GloVe internals
  - Building embeddings from scratch
  - Semantic similarity and vector operations
  - Dimensionality reduction and visualization
  - Using pre-trained embeddings


In [3]:
def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    """
    Calculate the cosine similarity between two numpy vectors.

    Formula: (A . B) / (||A|| * ||B||)
    """
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    # Handle zero vector edge case to prevent division by zero
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0

    return float(dot_product / (norm_v1 * norm_v2))

# Test cases
vec_a = np.array([1, 2, 3])
vec_b = np.array([1, 2, 3]) # Identical
vec_c = np.array([-1, -2, -3]) # Opposite
vec_d = np.array([0, 0, 0]) # Zero vector

print(f"Similarity (Identical): {cosine_similarity(vec_a, vec_b):.4f}")
print(f"Similarity (Opposite): {cosine_similarity(vec_a, vec_c):.4f}")
print(f"Similarity (Zero Vector): {cosine_similarity(vec_a, vec_d):.4f}")

Similarity (Identical): 1.0000
Similarity (Opposite): -1.0000
Similarity (Zero Vector): 0.0000


In [4]:
import numpy as np

class SimpleSkipGram:
    def __init__(self, corpus: list, embedding_dim: int = 8, window_size: int = 2, lr: float = 0.05):
        self.corpus = corpus
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        self.lr = lr

        # Build vocabulary
        self.words = sorted(list(set(" ".join(corpus).lower().split())))
        self.vocab_size = len(self.words)
        self.word_to_idx = {word: i for i, word in enumerate(self.words)}
        self.idx_to_word = {i: word for i, word in enumerate(self.words)}

        # Initialize embeddings: W (target vectors) and W_prime (context vectors)
        np.random.seed(42)
        self.W = np.random.uniform(-0.5, 0.5, (self.vocab_size, self.embedding_dim))
        self.W_prime = np.random.uniform(-0.5, 0.5, (self.vocab_size, self.embedding_dim))

    def _softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)

    def generate_training_data(self):
        training_pairs = []
        for sentence in self.corpus:
            tokens = sentence.lower().split()
            for i, target_word in enumerate(tokens):
                target_idx = self.word_to_idx[target_word]
                # Context window limits
                start = max(0, i - self.window_size)
                end = min(len(tokens), i + self.window_size + 1)

                for j in range(start, end):
                    if i != j:
                        context_idx = self.word_to_idx[tokens[j]]
                        training_pairs.append((target_idx, context_idx))
        return training_pairs

    def train(self, epochs: int = 100):
        pairs = self.generate_training_data()
        for epoch in range(epochs):
            loss = 0
            for target_idx, context_idx in pairs:
                # Forward pass
                # v_w is the target word representation
                v_w = self.W[target_idx]

                # Compute scores for all possible context words
                scores = np.dot(self.W_prime, v_w)
                predictions = self._softmax(scores)

                # Compute loss: -log(P(context_idx | target_idx))
                loss -= np.log(max(predictions[context_idx], 1e-15))

                # Backward pass & Gradient updates
                # Error: y_hat - y
                errors = predictions.copy()
                errors[context_idx] -= 1.0

                # Gradients
                dW_prime = np.outer(errors, v_w)
                dW = np.dot(self.W_prime.T, errors)

                # Update weights
                self.W_prime -= self.lr * dW_prime
                self.W -= self.lr * dW

            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} | Loss: {loss:.4f}")

    def get_word_vector(self, word: str) -> np.ndarray:
        idx = self.word_to_idx.get(word.lower())
        if idx is None:
            raise ValueError(f"Word '{word}' not in vocabulary.")
        # Combining target and context vectors for a richer representation
        return (self.W[idx] + self.W_prime[idx]) / 2.0

    def find_similarity(self, word1: str, word2: str) -> float:
        vec1 = self.get_word_vector(word1)
        vec2 = self.get_word_vector(word2)
        # Utilizing the previously defined global cosine_similarity function
        return cosine_similarity(vec1, vec2)

# Quick Demo
sample_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "kings wear crowns and queens wear jewelry",
    "artificial intelligence and machine learning are transforming science"
]

sg_model = SimpleSkipGram(corpus=sample_corpus, embedding_dim=5, window_size=2)
sg_model.train(epochs=100)

# Calculate similarities using our cosine_similarity function wrapper
sim_king_queen = sg_model.find_similarity("kings", "queens")
sim_fox_dog = sg_model.find_similarity("fox", "dog")
sim_learning_science = sg_model.find_similarity("learning", "science")

print("\n--- Semantic Similarity Results ---")
print(f"Similarity ('kings', 'queens'): {sim_king_queen:.4f}")
print(f"Similarity ('fox', 'dog'): {sim_fox_dog:.4f}")
print(f"Similarity ('learning', 'science'): {sim_learning_science:.4f}")

Epoch 1/100 | Loss: 240.5739
Epoch 20/100 | Loss: 220.8276
Epoch 40/100 | Loss: 221.2809
Epoch 60/100 | Loss: 222.9691
Epoch 80/100 | Loss: 218.0463
Epoch 100/100 | Loss: 213.9017

--- Semantic Similarity Results ---
Similarity ('kings', 'queens'): -0.2334
Similarity ('fox', 'dog'): 0.0034
Similarity ('learning', 'science'): 0.7314


---
## Part 1: Introduction to Word2Vec and GloVe internals

Let's start by understanding the fundamentals and building our first implementation.

In [2]:
# Example implementation
# This is a starter template - expand with your implementation

class ExampleClass:
    """
    Example class for Word2Vec and GloVe internals.
    """

    def __init__(self):
        self.data = []

    def process(self, input_data):
        """
        Process the input data.
        """
        # Add your implementation here
        return input_data

# Test the class
example = ExampleClass()
result = example.process('test data')
print(f'Result: {result}')

Result: test data


### 🎯 Exercise 1 (Easy): Getting Started

**Task**: Implement a basic version of the concept introduced above.

**Requirements**:
1. Follow the pattern shown in the example
2. Test your implementation with sample data
3. Verify the output matches expectations
4. Add error handling for edge cases

In [5]:
class WordFrequencyCounter:
    """
    A class to calculate word frequencies in a given corpus of text
    following the template patterns from Part 1.
    """
    def __init__(self):
        self.word_counts = {}

    def process(self, input_text: str) -> dict:
        """
        Clean the input text and update word frequency counts.
        """
        if not isinstance(input_text, str):
            raise TypeError("Input data must be a string.")

        # Lowercase and strip punctuation
        clean_text = re.sub(r'[^\w\s]', '', input_text.lower())
        words = clean_text.split()

        for word in words:
            self.word_counts[word] = self.word_counts.get(word, 0) + 1

        return self.word_counts

# Test our implementation
counter = WordFrequencyCounter()

# Test case 1: Basic sentence
result_1 = counter.process("The quick brown fox jumps over the lazy dog.")
print("Test 1 (Basic Sentence):")
print(result_1)

# Test case 2: Overlapping vocabulary
result_2 = counter.process("More foxes, more dogs, and more jumpings!")
print("\nTest 2 (Cumulative counts & punctuation handling):")
print(result_2)

Test 1 (Basic Sentence):
{'the': 2, 'quick': 1, 'brown': 1, 'fox': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1}

Test 2 (Cumulative counts & punctuation handling):
{'the': 2, 'quick': 1, 'brown': 1, 'fox': 1, 'jumps': 1, 'over': 1, 'lazy': 1, 'dog': 1, 'more': 3, 'foxes': 1, 'dogs': 1, 'and': 1, 'jumpings': 1}


---
## Part 2: Building embeddings from scratch

Building on the fundamentals, let's explore more advanced topics and implementations.

In [ ]:
# Advanced implementation example

def advanced_function(data: List[Any], **kwargs) -> Dict[str, Any]:
    """
    Advanced function for Building embeddings from scratch.

    Args:
        data: Input data to process
        **kwargs: Additional parameters

    Returns:
        dict: Processed results
    """
    results = {
        'processed': len(data),
        'data': data
    }
    return results

# Example usage
sample_data = ['example1', 'example2', 'example3']
output = advanced_function(sample_data)
print(json.dumps(output, indent=2))

### 🎯 Exercise 2 (Intermediate): Building on the Basics

**Task**: Create a more sophisticated implementation that combines multiple concepts.

**Requirements**:
1. Use the patterns from Part 1 and Part 2
2. Add parameter validation
3. Include helpful error messages
4. Test with various input types

In [ ]:
# YOUR CODE HERE
# Implement the intermediate exercise

pass  # Replace with your implementation

---
## Part 3: Semantic similarity and vector operations

Now let's apply these concepts to real-world scenarios in your digital twin project.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt

def visualize_results(data: List[Any]):
    """
    Visualize the processing results.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Add your visualization code here
    ax.set_title('Results Visualization')
    ax.set_xlabel('X-axis')
    ax.set_ylabel('Y-axis')

    plt.tight_layout()
    plt.show()

# Example: visualize_results(sample_data)

### 🎯 Exercise 3 (Advanced): Complete Implementation

**Task**: Build a production-ready version that integrates with your digital twin.

**Requirements**:
1. Combine all concepts from this notebook
2. Add comprehensive error handling
3. Include logging and debugging capabilities
4. Create visualization of results
5. Write documentation for your implementation

In [ ]:
# YOUR CODE HERE
# Build your complete implementation

class ProductionSystem:
    """
    Production-ready implementation for digital twin.
    """

    def __init__(self):
        # Initialize your system
        pass

    def process(self, input_data):
        # Add your implementation
        pass

# Test your production system
# Add comprehensive tests here

---
## Summary & Key Takeaways

### What We Learned:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

### Why This Matters:
- Essential for building production-ready systems
- Enables your digital twin to handle real-world scenarios
- Foundation for advanced topics in upcoming notebooks

### Next Steps:
1. Complete all exercises in this notebook
2. Integrate concepts with your digital twin project
3. Test your implementation thoroughly
4. Move on to the next notebook in Week 2

## 🏆 Challenge Project

**Build a Complete Word2Vec and GloVe internals System for Your Digital Twin**

**Requirements**:
1. Apply all concepts from this notebook
2. Handle edge cases and errors gracefully
3. Include comprehensive testing
4. Create visualizations of your results
5. Document your implementation
6. Integrate with your existing digital twin code

**Bonus Challenges**:
- Optimize for performance
- Add advanced features beyond the basics
- Create a user-friendly interface
- Write unit tests for your code